In [14]:
import cv2
import pandas as pd
from datetime import datetime

In [18]:
names = {
    0:"Animesh"
}

In [19]:
df = pd.DataFrame(columns=["Name","Date","Time","Status"])
df.to_csv("Attendance.csv",index=False)

In [ ]:
# Load the trained model
recognizer = cv2.face.LBPHFaceRecognizer_create()
recognizer.read("trainer/trainer.yml")

faceCascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

attendance_file = "Attendance.csv"

try:
    attendance_df = pd.read_csv(attendance_file)
except:
    attendance_df = pd.DataFrame(columns=["Name","Date","Time","Status"])
    
# Start video capture
cap = cv2.VideoCapture(0)

while True:
    
    ret, img = cap.read()
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    faces = faceCascade.detectMultiScale(gray,1.2,5)
    
    for(x,y,w,h) in faces:
        
        id,confidence = recognizer.predict(gray[y:y+h,x:x+w])
        
        if confidence < 80:
            
            name = names.get(id,"Unknown")
            
            now = datetime.now()
            
            date = now.strftime("%d-%m-%Y")
            time = now.strftime("%H:%M:%S")
            
            # Prevent duplicate attendance
            if not ((attendance_df["Name"] == name) & (attendance_df["Date"] == date)).any():
                
                data = [[name,date,time,"Present"]]
                
                new_entry = pd.DataFrame(data,columns=["Name","Date","Time","Status"])
                
                new_entry.to_csv("Attendance.csv",mode='a',header=False,index=False)
                
                attendance_df = pd.concat([attendance_df,new_entry],ignore_index=True)
                
                print(name,"Attendance Recorded")
                
        else:
            name = "Unknown"
        
        cv2.rectangle(img,(x,y),(x+w,y+h),(0,255,0),2)
        
        cv2.putText(img,str(name),(x,y-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,(255,255,255),2)
    
    cv2.imshow("AI Attendance System",img)
    
    if cv2.waitKey(1)==13:
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
# Remove duplicate attendance entries

df = pd.read_csv("Attendance.csv")

df = df.drop_duplicates(subset=["Name","Date"], keep="first")

df.to_csv("Attendance.csv", index=False)

print("Duplicate attendance cleaned")

Duplicate attendance cleaned
